# Correlation clustering: returns versus normalized FX levels

Can currencies share a forecasting model? Compare two representations over exactly **January 2011–December 2019**:

- **Monthly log returns:** groups currencies that move together in the same months.
- **Normalized FX levels:** groups currencies with similar exchange-rate paths. This is a valid alternative; it answers a different question.

Use the same currencies, dates, Pearson correlation distance and average-linkage hierarchical algorithm for both. There is no time warping or random initialization. This notebook is independent of the DTW notebook and does not change its results.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.manifold import MDS

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "reports" / "clustering" / "correlation"
OUT.mkdir(parents=True, exist_ok=True)
START, END = "2011-01", "2019-12"
K_VALUES = range(2, 7)

## 1. Keep the sample identical

Select exact BIS FX source IDs and apply the existing currency-launch and peg masks before calculating returns. Keep missing observations missing. A currency enters both comparisons only if its levels and returns are complete and nonconstant throughout the common calendar.

FX is local currency per USD. A positive return therefore means local-currency depreciation. January 2011's return also uses the observed December 2010 FX level.

In [2]:
cfg = json.loads((ROOT / "config/research_sample.json").read_text())
metadata = pd.DataFrame(cfg["currencies"]).set_index("currency")
obs = pd.read_parquet(ROOT / "data/ingested/observations.parquet")
sources = {c["source_series"]["fx"]: c["currency"] for c in cfg["currencies"]}
rows = obs.loc[obs.dataset.eq("WS_XRU") & obs.frequency.eq("M")
               & obs.series_id.isin(sources)].copy()
rows["currency"] = rows.series_id.map(sources)
fx = rows.pivot(index="period", columns="currency", values="value")
fx.index = pd.PeriodIndex(fx.index, freq="M")
fx = fx.reindex(index=pd.period_range(**cfg["monthly_window"], freq="M"),
                columns=metadata.index).astype(float)
fx = fx.where(np.isfinite(fx) & fx.gt(0))
for c in cfg["currencies"]:
    fx.loc[fx.index < pd.Period(c["first_eligible_month"], "M"), c["currency"]] = np.nan
area_to_currency = {c["area"]: c["currency"] for c in cfg["currencies"]}
for interval in cfg["excluded_month_intervals"]:
    fx.loc[interval["start"]:interval["end"], area_to_currency[interval["area"]]] = np.nan

In [3]:
returns = (100 * np.log(fx).diff()).loc[START:END]
levels = fx.loc[START:END]
coverage = pd.DataFrame({"missing_returns": returns.isna().sum(),
                         "missing_levels": levels.isna().sum()})
coverage["included"] = (coverage.eq(0).all(axis=1)
                         & returns.std(ddof=0).gt(1e-12) & levels.std(ddof=0).gt(1e-12))
currencies = coverage.index[coverage.included].tolist()
returns, levels = returns[currencies], levels[currencies]
display(coverage)

,missing_returns,missing_levels,included
currency,,,
DZD,0,0,True
ARS,0,0,True
AUD,0,0,True
BAM,0,0,True
BRL,0,0,True
CAD,0,0,True
CLP,0,0,True
CNY,0,0,True
TWD,0,0,True


**Saved-run check:** 39 currencies and 108 calendar months in both comparisons, with no filled FX values.

## 2. What normalization changes

For each currency, normalize its FX levels as \(z_t=(S_t-ar S)/s_S\), using only the historical clustering window. This removes arbitrary currency units, mean and scale. It does **not** remove long-term trends.

Pearson correlation is already invariant to a positive linear rescaling: raw, rebased-to-100 and z-scored levels have the same correlation matrix on the same dates. The substantive change is **levels versus returns**, not the choice to z-score levels. Correlation of z-scored levels is also equivalent to scaled Euclidean distance between those standardized paths.

For both representations use \(d_{ij}=\sqrt{2(1-ho_{ij})}\). High positive correlation means small distance; negative correlation means large distance. Average linkage merges groups using their average pairwise distance. Hierarchical clustering has no k-means inertia, so inspect the silhouette, dendrogram and group sizes.

In [4]:
normalized_returns = (returns - returns.mean()) / returns.std(ddof=0)
normalized_levels = (levels - levels.mean()) / levels.std(ddof=0)
representations = {"returns": normalized_returns, "levels": normalized_levels}
titles = {"returns": "Monthly return correlation", "levels": "Normalized FX levels"}
correlations, distances, trees, labels_by_k, records = {}, {}, {}, {}, []
for name, values in representations.items():
    correlations[name] = values.corr()
    distance = np.sqrt(np.clip(2 * (1 - correlations[name].to_numpy()), 0, None))
    np.fill_diagonal(distance, 0)
    distances[name] = distance
    trees[name] = linkage(squareform(distance, checks=False), method="average")
    for k in K_VALUES:
        labels = fcluster(trees[name], k, criterion="maxclust")
        labels_by_k[name, k] = labels
        sizes = pd.Series(labels).value_counts().sort_values()
        records.append({"representation": name, "k": k,
                        "silhouette": silhouette_score(distance, labels, metric="precomputed"),
                        "smallest_cluster": sizes.min(), "largest_cluster": sizes.max(),
                        "cluster_sizes": " + ".join(sizes.astype(str))})
selection = pd.DataFrame(records)
display(selection.round(3))

,representation,k,silhouette,smallest_cluster,largest_cluster,cluster_sizes
0,returns,2,0.243,1,38,1 + 38
1,returns,3,0.219,1,36,1 + 2 + 36
2,returns,4,0.182,1,35,1 + 1 + 2 + 35
3,returns,5,0.142,1,34,1 + 1 + 1 + 2 + 34
4,returns,6,0.112,1,33,1 + 1 + 1 + 1 + 2 + 33
5,levels,2,0.551,2,37,2 + 37
6,levels,3,0.401,1,36,1 + 2 + 36
7,levels,4,0.362,1,34,1 + 2 + 2 + 34
8,levels,5,0.352,1,34,1 + 1 + 1 + 2 + 34
9,levels,6,0.314,1,29,1 + 1 + 1 + 2 + 5 + 29


In [5]:
fig = go.Figure()
for name, table in selection.groupby("representation"):
    fig.add_scatter(x=table.k, y=table.silhouette, mode="lines+markers", name=titles[name],
                    customdata=table.cluster_sizes,
                    hovertemplate="K=%{x}<br>Silhouette: %{y:.3f}<br>Group sizes: %{customdata}<extra>%{fullData.name}</extra>")
fig.update_layout(template="plotly_white", title="Silhouette and group sizes, K=2–6",
                  xaxis_title="Number of clusters", yaxis_title="Silhouette", height=360)
fig.update_xaxes(dtick=1)
fig.show()

## 3. Choose K without forcing useful-looking groups

**Use K=2 for each representation**, the highest silhouette in the inspected K=2–6 range. This selects a descriptive split, not a validated training design.

| Representation | Silhouette at K=2 | Group sizes | Reading |
| --- | ---: | --- | --- |
| Monthly returns | 0.243 | 38 + 1 | JPY separates from the rest. |
| Normalized FX levels | 0.551 | 37 + 2 | ISK and ILS separate from the rest. |

Both mostly separate a small group from a large one. The higher levels score describes cleaner separation **in level-path space**; it is not evidence of better forecasts. Shared trends can make level correlations large. A singleton silhouette is zero by convention.

For comparison with the earlier three-group DTW result, the table above also reports K=3: returns produce 36 + 2 + 1, and levels produce 36 + 2 + 1. Cluster sizes remain important even if a silhouette score looks favorable.

In [6]:
chosen_k = {"returns": 2, "levels": 2}
memberships, summaries = {}, []
for name in representations:
    labels = labels_by_k[name, chosen_k[name]]
    size_order = pd.Series(labels).value_counts().index
    numbering = {label: i + 1 for i, label in enumerate(size_order)}
    members = pd.DataFrame(index=pd.Index(currencies, name="currency"))
    members["economy"] = metadata.loc[currencies, "economy"]
    members["pool_id"] = [numbering[label] for label in labels]
    members["silhouette"] = silhouette_samples(distances[name], labels, metric="precomputed")
    memberships[name] = members
    for pool_id, group in members.groupby("pool_id"):
        summaries.append({"representation": name, "pool_id": pool_id, "currencies": len(group),
                          "silhouette": group.silhouette.mean(), "members": ", ".join(sorted(group.index))})
summary = pd.DataFrame(summaries)
display(summary.round(3))

,representation,pool_id,currencies,silhouette,members
0,returns,1,38,0.250,"ARS, AUD, BAM, BRL, CAD, CHF, CLP, CNY, COP, C..."
1,returns,2,1,0.000,JPY
2,levels,1,37,0.561,"ARS, AUD, BAM, BRL, CAD, CHF, CLP, CNY, COP, C..."
3,levels,2,2,0.360,"ILS, ISK"


## 4. Plot the groups together

Each map is an MDS projection of its own distance matrix, colored by that representation's groups. Axes are for visualization only; positions are not directly comparable across the two maps. Hover to identify a currency. The standardized-history plots use actual calendar dates, without warping.

In [7]:
maps = {}
for name in representations:
    xy = MDS(n_components=2, dissimilarity="precomputed", random_state=42,
             n_init=8, max_iter=1000, eps=1e-6, normalized_stress=False).fit_transform(distances[name])
    maps[name] = memberships[name].join(pd.DataFrame(xy, index=currencies, columns=["x", "y"]))

fig = make_subplots(rows=1, cols=2, subplot_titles=list(titles.values()))
for col, name in enumerate(representations, 1):
    for pool_id, group in maps[name].groupby("pool_id"):
        fig.add_trace(go.Scatter(x=group.x, y=group.y, mode="markers+text", text=group.index,
                      textposition="top center", textfont=dict(size=9), name=f"{name}: group {pool_id}",
                      marker=dict(color={1: "#2B6CB0", 2: "#B77916"}[pool_id], size=9),
                      hovertemplate="%{text}<extra>%{fullData.name}</extra>"), row=1, col=col)
fig.update_layout(template="plotly_white", title="Same currencies and dates; different representation",
                  height=460, showlegend=False)
fig.show()

In [8]:
fig = make_subplots(rows=2, cols=1, subplot_titles=list(titles.values()))
for row, (name, values) in enumerate(representations.items(), 1):
    for pool_id, group in memberships[name].groupby("pool_id"):
        for currency in group.index:
            fig.add_trace(go.Scatter(x=values.index.to_timestamp(), y=values[currency],
                          mode="lines", line=dict(color={1: "#2B6CB0", 2: "#B77916"}[pool_id], width=1),
                          opacity=.4 if len(group) > 2 else .9, name=currency,
                          hovertemplate="%{fullData.name}<br>%{x|%b %Y}: %{y:.2f} SD<extra></extra>"), row=row, col=1)
fig.update_layout(template="plotly_white", title="Standardized histories, grouped by color",
                  height=650, showlegend=False)
fig.update_yaxes(title_text="Standard deviations")
fig.show()

## 5. Read the hierarchy

A dendrogram shows when currencies or groups merge. The dashed line cuts each tree at K=2. There is no requirement that the resulting groups have similar sizes. Small outlier groups are a finding to report, not a reason to hide observations or force K=3.

In [9]:
fig = make_subplots(rows=2, cols=1, subplot_titles=list(titles.values()))
for row, name in enumerate(representations, 1):
    tree = dendrogram(trees[name], labels=currencies, no_plot=True)
    for x, y in zip(tree["icoord"], tree["dcoord"]):
        fig.add_trace(go.Scatter(x=x, y=y, mode="lines", line=dict(color="#63778C", width=1),
                                hoverinfo="skip", showlegend=False), row=row, col=1)
    cut = (trees[name][-1, 2] + trees[name][-2, 2]) / 2
    fig.add_hline(y=cut, line_dash="dot", line_color="#B77916", row=row, col=1)
    fig.update_xaxes(tickvals=np.arange(5, 10 * len(currencies), 10), ticktext=tree["ivl"],
                     tickangle=-65, tickfont=dict(size=9), row=row, col=1)
fig.update_layout(template="plotly_white", height=650, title="Average-linkage dendrograms; dotted line = K=2")
fig.update_yaxes(title_text="Merge distance")
fig.show()

## 6. Compare with DTW and keep the interpretation tied to the project

The next table compares memberships with the saved DTW grouping. ARI compares membership assignments, not forecasting performance. It can also change because the selected number of groups differs.

**Why use normalized levels?** They are a legitimate description of historical FX paths. If the question is which currencies share long-term depreciation or appreciation patterns, level-based clustering is relevant. If the question is contemporaneous responses to common shocks, return correlation is more direct. Neither establishes that the countries have the same relationship between macro predictors and future returns.

For pooled training, retain global pooling as the benchmark. These K=2 results do not yet identify several substantial, distinct training pools. Validate any group indicator or separate-model design on later dates. All normalization and grouping must use only information available before evaluation.

In [10]:
dtw = pd.read_csv(ROOT / "reports/clustering/memberships.csv").set_index("currency")
comparisons = pd.DataFrame({
    "comparison": ["Returns vs normalized levels", "Returns vs saved DTW", "Levels vs saved DTW"],
    "ARI": [adjusted_rand_score(memberships["returns"].pool_id, memberships["levels"].pool_id),
            adjusted_rand_score(memberships["returns"].pool_id, dtw.loc[currencies, "pool_id"]),
            adjusted_rand_score(memberships["levels"].pool_id, dtw.loc[currencies, "pool_id"])]})
display(comparisons.round(3))

,comparison,ARI
0,Returns vs normalized levels,-0.034
1,Returns vs saved DTW,-0.021
2,Levels vs saved DTW,-0.037


## 7. Save the comparison

Save only the inputs needed to reproduce and inspect this comparison: membership, K scores, the actual standardized histories, distances, projections and settings. Group IDs are categorical. Groups with one or two currencies are marked descriptively; no separate forecasting models are trained here.

In [11]:
coverage.to_csv(OUT / "coverage.csv")
selection.to_csv(OUT / "model_selection.csv", index=False)
summary.to_csv(OUT / "pool_summary.csv", index=False)
comparisons.to_csv(OUT / "comparisons.csv", index=False)
for name, values in representations.items():
    memberships[name].to_csv(OUT / f"{name}_memberships.csv")
    values.to_csv(OUT / f"{name}_histories.csv")
    maps[name].to_csv(OUT / f"{name}_map.csv")
    correlations[name].to_csv(OUT / f"{name}_correlations.csv")
    pd.DataFrame(distances[name], index=currencies, columns=currencies).to_csv(OUT / f"{name}_distances.csv")
    pd.DataFrame(trees[name], columns=["left", "right", "distance", "count"]).to_csv(OUT / f"{name}_tree.csv", index=False)
settings = {"fit_start": START, "fit_end": END, "currencies": currencies, "months": len(returns),
            "distance": "sqrt(2 * (1 - Pearson correlation))", "linkage": "average", "chosen_k": chosen_k,
            "level_normalization": "Within-currency z-score of raw FX levels, training window only"}
(OUT / "settings.json").write_text(json.dumps(settings, indent=2))

743

**Takeaway:** clustering normalized currencies is valid, and in this sample separates historical paths more cleanly by silhouette. That does not settle which grouping helps the forecasting task. Both correlation approaches mainly isolate a small group; the next decision should depend on later forecast performance, not silhouette alone.

The numerical statements in Markdown describe the saved default run; review them when changing dates or settings.

Methods: [SciPy average linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html), [silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).